# fMRI encoding analysis
Build voxel-wise encoding models linking unit and voxel responses using non-negative ridge regression.

1. Which unit type explains the most variance in each voxel type?
    - Fit separate models
2. How much unique vs. shared variance does each unit type explain in each voxel type?
    - Fit joint models and apply variance partitioning


In [ ]:
from facebody.config import PROJECT_ROOT, SUBJECTS
from facebody.model_sets import TRAINED, ENCODING_MODEL, ENCODING_LAYERS
from facebody.dnn.floc import stats_floc_dprime_summary, compute_mean_resp_nsd
from facebody.fmri.streams import ROIS, ROIS_COMPLETE, extract_streams_resp
from facebody.encoding import (
    VoxelwiseEncodingPipeline, VoxelwiseEncodingConfig,
    stats_pairwise_comps_sep, stats_sep_dd, stats_varpart_dd,
)
from facebody.plotting import (
    plot_top_nsd_imgs,
    plot_encoding_sep, plot_encoding_sep_layers,
    plot_encoding_varpart, plot_encoding_varpart_layers,
    plot_integration_ventral, stats_integration_ventral,
)
from facebody.plotting.flatmaps import (
    plot_encoding_sep_wta_flatmap, plot_encoding_sep_wta_flatmaps_subjects,
    plot_encoding_varpart_flatmaps, plot_encoding_varpart_flatmaps_subjects,
)
from facebody.myutils.nsd import (
    download_sessions, prepro_img_meta, prepro_resp,
    define_rois, save_subj_img_ids,
)

device = "cuda:0"

## Top images per unit type

In [ ]:
# Models
print(f"Trained (N={len(TRAINED)}): {list(TRAINED)}")

In [ ]:
# Compute mean response per unit type
compute_mean_resp_nsd(TRAINED, device)

In [ ]:
# Fig. S8: Plot top NSD images per unit type averaged across models
plot_top_nsd_imgs(TRAINED, out_path="suppl/top_nsd_images.pdf")

## Voxel-wise encoding models
- Natural Scenes Dataset (NSD; Allen et al., 2022)
    - N=8 participants
    - Visual cortex based on NSD streams mask
    - Face-selective, body-selective, and overlap ROIs defined using same fLoc

In [ ]:
# Model, layers, and voxel types
MODEL = ENCODING_MODEL
LAYERS = ENCODING_LAYERS
LAYER = LAYERS[-1]
VOX_TYPES = ["face", "body", "mixed", "non"]

In [ ]:
# Check selectivity of matched unit types
stats_floc_dprime_summary(MODEL, LAYERS)

### Preprocess fMRI data

In [ ]:
# Download and preprocess fMRI data
download_sessions(SUBJECTS)
prepro_img_meta()
prepro_resp(SUBJECTS)
define_rois(SUBJECTS, PROJECT_ROOT)
extract_streams_resp(SUBJECTS, ROIS, ROIS_COMPLETE)
save_subj_img_ids(SUBJECTS, PROJECT_ROOT)

### Fit and evaluate encoding models

In [ ]:
# Fit voxel-wise ridge regression
cfg = VoxelwiseEncodingConfig(
    MODEL, LAYERS, SUBJECTS,
    device=device,
    perm_layers=("fc7",),
    n_perm=1000,
)
en = VoxelwiseEncodingPipeline(cfg)
en.ensure_activs(save_to_disk=True)
en.ensure_nmf(force=False)
en.run()

### Prediction performance

In [ ]:
# Fig. 3B: Explained variance of each unit type per voxel type
plot_encoding_sep(
    MODEL, LAYER, SUBJECTS, VOX_TYPES,
    out_path="main/encoding_sep_sel.pdf",
)

for vox_type in VOX_TYPES:
    stats_pairwise_comps_sep(MODEL, LAYER, [vox_type])
stats_sep_dd(MODEL, LAYER)

stats_pairwise_comps_sep(MODEL, LAYER, ["OPA", "PPA"])

In [ ]:
# Fig. 3C: Flatmap with winning unit type based on explained variance per voxel (participant S2)
plot_encoding_sep_wta_flatmap(
    "subj02", MODEL, LAYER,
    out_path="main/encoding_sep_r2_wta_flatmap.pdf",
)

In [ ]:
# Fig. S9: Flatmap with winning unit type based on explained variance per voxel (all participants)
plot_encoding_sep_wta_flatmaps_subjects(
    SUBJECTS, MODEL, LAYER,
    out_path="suppl/encoding_sep_r2_wta_flatmap_subjects.pdf",
)

In [ ]:
# Fig. S10: Explained variance of each unit type per ROI
plot_encoding_sep(
    MODEL, LAYER, SUBJECTS, ROIS,
    r2_adj=False, ylim=(0, 0.4),
    out_path="suppl/encoding_sep_rois.pdf",
)

In [ ]:
# Fig. S11: Explained variance of each unit type across layers
plot_encoding_sep_layers(
    MODEL, LAYERS, SUBJECTS, ROIS,
    r2_adj=False,
    out_path="suppl/encoding_sep_layers.pdf",
)

### Variance partitioning

In [ ]:
# Fig. 4B: Variance portion of each unit type per voxel type
plot_encoding_varpart(
    MODEL, LAYER, SUBJECTS, VOX_TYPES,
    out_path="main/encoding_varpart_sel.pdf",
)
stats_varpart_dd(MODEL, LAYER)

In [ ]:
# Fig. 4C: Flatmap with variance portion per unit type (participant S2)
plot_encoding_varpart_flatmaps(
    "subj02", MODEL, LAYER,
    out_path="main/encoding_varpart_r2_flatmaps.pdf",
)

In [ ]:
# Fig. S12-13: Flatmap with variance portion per unit type (all participants)
plot_encoding_varpart_flatmaps_subjects(
    SUBJECTS, MODEL, LAYER,
    out_path="suppl/encoding_varpart_r2_flatmaps_subjects.pdf",
)

In [ ]:
# Fig. S14: Variance portion of each unit type per ROI
plot_encoding_varpart(
    MODEL, LAYER, SUBJECTS, ROIS,
    out_path="suppl/encoding_varpart_rois.pdf",
)

In [ ]:
# Fig. S15: Variance partitioning of unit types across layers
plot_encoding_varpart_layers(
    MODEL, LAYERS, SUBJECTS, ROIS,
    ylim=(0, 0.25), out_path="suppl/encoding_varpart_layers.pdf",
)

In [ ]:
# Fig. 4D: Integration across ventral pathway
plot_integration_ventral(
    MODEL, LAYER, SUBJECTS,
    out_path="main/encoding_integration_ventral.pdf",
)
stats_integration_ventral(MODEL, SUBJECTS, LAYER)